# Example Zernike Projection

This notebook demonstrates a way to compute the Zernike coefficients: <br>
Remember that an arbitrary wavefront can be described as a linear combination of Zernike (circular) Polynomials.

\begin{equation}
    W(r,\theta) = \sum_{i=1}^{\infty}a_iZ_i(r,\theta)
\end{equation}

Where $a_i$ = Coefficient and $Z_i(r,\theta)$ is the Zernike Polynomial <br>
As an intuition, the coefficient $a_i$ tells you how much of the abberation mode $Z_i(r,\theta)$ is present within the wavefront $W(r,\theta)$. For example, in the example_zernike_projection.m file from J. Schmidt, an arbitrary wavefront with 3 modes:

$$
    W(r,\theta) = 0.5Z_2(r,\theta) + 0.25Z_4(r,\theta) - 0.6Z_{21}(r,\theta)
$$

What does this mean? This means that the wavefront contains 0 piston, 0.5 amount of $x$ tilt, 0 $y$ tilt, 0.25 amount of defocus, ... ,  -0.6 amount of $y$ pentafoil, ...<br>
Refer to Table 5.2 to look up the different modes.

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import pinv
from optprop.aperture_functions import circ
from optprop.transforms import cart2pol
from optprop.zernike import zernike

N = 32
L = 2
delta = L/N
x = np.arange(-N/2,N/2,1) * delta
x,y = np.meshgrid(x,x)
r,theta = cart2pol(x,y)

ap = circ(x,y,2)
z2 = zernike(2,r,theta) * ap
z4 = zernike(4,r,theta) * ap
z21 = zernike(21,r,theta) * ap

W = 0.5 * z2 + 0.25 * z4 - 0.6 * z21    # Arbitrary Wavefront Abberation

idx = ap.astype(bool)
W = (W.T[idx.T])[:,None]                # Reshape 1D vector into 2D Column vector from (795,) to (795,1)
z2_col = (z2.T[idx.T])[:,None]          # z2.T[idx.T] does indexing in the column-major order. i.e. It creates a 1D vector by reaching from top to bottom.
z4_col = (z4.T[idx.T])[:,None]
z21_col = (z21.T[idx.T])[:,None]
Z = np.c_[z2_col,z4_col,z21_col]
Z_inv = pinv(Z)
A = Z_inv @ W                           # Regular numpy.linalg.inv doesn't work since Z is a non-square matrix. Use pinv for Moore-Penrose pseudo inverse matrix.
print(f"A = inv(Z) * W\nA = \n{A}")


A = inv(Z) * W
A = 
[[ 0.5 ]
 [ 0.25]
 [-0.6 ]]
